# Compensation Mechanisms Study — Phase 3 (FP32 & QAT INT8)

**Author:** Rafael  
**Dataset:** Tiny-ImageNet-200 (200 classes, 64×64 RGB)  
**Goal:** Train and compare all Phase 3 compensation mechanisms in FP32, then apply
Quantization-Aware Training (QAT) to the QAT-compatible models to obtain INT8 versions.
Report **Top-1** and **Top-5** accuracy for every model.

| Model | Mechanism | QAT | Notes |
|---|---|---|---|
| AlexNetBottleneck | 1×1→3×3→1×1 bottleneck | ✓ | Conv-BN-ReLU triples; nested blocks |
| AlexNetFactorized | 1×3 + 3×1 asymmetric | ✓ | Inception-style spatial factorization |
| AlexNetGroupConv | Grouped convolutions (g=4) | ✓ | 4× fewer cross-channel params |
| AlexNetDepthwiseSep | Depthwise separable | ✓ | MobileNet-style DW+PW pairs |
| AlexNetResidual | Residual skip connections | ✓ | FloatFunctional add; most impactful |
| AlexNetFire | Fire modules (SqueezeNet) | ✓ | torch.cat; nested squeeze+expand |
| AlexNetSE | Squeeze-and-Excitation | ✗ | Sigmoid not fbgemm-compatible |

Note: the GAP-head variant (formerly `AlexNetGAP` here) now lives in Phase 2's
`alexnet_variants.py` as `AlexNet3x3GAP`, alongside its FC-head counterpart. Its
historical results remain in `results.csv` under the `alexnet_gap` run name.

## Notebook layout

1. Imports & reproducibility
2. Dataset & loaders
3. Model shape check
4. Model registration (fuse maps)
5. FP32 training
6. QAT training (all except AlexNetSE)
7. INT8 conversion & CPU evaluation
8. FP32 evaluation (reload best checkpoints)
9. Final comparison table
10. Persist experiment summary

## 1. Imports & reproducibility

In [ ]:
import json
import os
import random
import sys
from dataclasses import asdict, replace
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torchinfo
import wandb

project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

from ml import (
    DataConfig, TrainerConfig, QATConfig,
    create_imagenet_loaders,
    MODEL_REGISTRY, register_model,
    Trainer, make_qat_callback,
    build_qat, build_comparison_table,
    load_best_model, convert_to_int8,
    find_fuse_groups,
    auto_resume_path, compress_checkpoint,
    create_results_summary, disk_mb, gzip_mb,
    compute_flops, make_run_summary,
)
from configs.loader import load_config

from models import (
    AlexNetBottleneck,
    AlexNetFactorized,
    AlexNetGroupConv,
    AlexNetDepthwiseSep,
    AlexNetResidual,
    AlexNetFire,
    AlexNetFireBypass,
    AlexNetSE,
)

torch.backends.quantized.engine = "fbgemm"

In [2]:
SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

SAVE_DIR    = project_root / "checkpoints" / "compensation_phase3"
RESULTS_DIR = project_root / "results" / "phase_3_compensation_and_hybrids_training"
SAVE_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

data_cfg = DataConfig(**load_config("data.yaml"))
data_cfg.seed = SEED
fp32_cfg = TrainerConfig(**load_config("training.yaml"))
qat_cfg  = QATConfig(**load_config("qat.yaml"))
print(device)

cuda


## 2. Dataset & loaders

In [3]:
import kagglehub

dataset_path = kagglehub.dataset_download("akash2sharma/tiny-imagenet")
train_path = os.path.join(dataset_path, "tiny-imagenet-200", "train")
data_cfg.dataset_path = train_path
print("Tiny-ImageNet train path:", train_path)

train_ds, val_ds, train_loader, val_loader = create_imagenet_loaders(data_cfg)

print(f"Train samples: {len(train_ds):,}")
print(f"Val   samples: {len(val_ds):,}")
print(f"Classes:       {data_cfg.num_classes}")
print(f"Batches/epoch: train={len(train_loader)}, val={len(val_loader)}")

Tiny-ImageNet train path: /home/rafael/.cache/kagglehub/datasets/akash2sharma/tiny-imagenet/versions/1/tiny-imagenet-200/train
Train samples: 90,000
Val   samples: 10,000
Classes:       200
Batches/epoch: train=1406, val=157


## 3. Model shape check

Seven Phase 3 compensation mechanisms — all from `models/compensation.py`:

| Name | Mechanism | BN? | Head |
|---|---|---|---|
| AlexNetBottleneck | 1×1→3×3→1×1 bottleneck | Yes | Linear(256, 200) |
| AlexNetFactorized | 1×3 + 3×1 factorized | Yes | 3-layer MLP |
| AlexNetGroupConv | Grouped conv (g=4) | Yes | 3-layer MLP |
| AlexNetDepthwiseSep | Depthwise separable | Yes | Linear(256, 200) |
| AlexNetResidual | Residual connections | Yes | 3-layer MLP |
| AlexNetFire | Fire modules | Yes | Linear(256, 200) |
| AlexNetSE | Squeeze-and-Excitation | No | 3-layer MLP |

In [ ]:
x = torch.randn(2, 3, data_cfg.img_size, data_cfg.img_size)
for label, ctor in [
    ("AlexNetBottleneck",   AlexNetBottleneck),
    ("AlexNetFactorized",   AlexNetFactorized),
    ("AlexNetGroupConv",    AlexNetGroupConv),
    ("AlexNetDepthwiseSep", AlexNetDepthwiseSep),
    ("AlexNetResidual",     AlexNetResidual),
    ("AlexNetFire",         AlexNetFire),
    ("AlexNetFireBypass",   AlexNetFireBypass),
    ("AlexNetSE",           AlexNetSE),
]:
    m = ctor().eval()
    with torch.no_grad():
        y = m(x)
    assert y.shape == (2, data_cfg.num_classes), f"{label}: unexpected shape {y.shape}"

## 4. Model registration

Fuse maps for flat-Sequential models are hand-written; nested block models
(Bottleneck, Residual, Fire) use `find_fuse_groups()` for auto-detection.
`AlexNetSE` is excluded from QAT — Sigmoid in SE blocks is not fbgemm-compatible.

In [ ]:
# ── Nested block models — auto-detect Conv-BN(-ReLU) groups ──────────────────
FUSE_BOTTLENECK  = find_fuse_groups(AlexNetBottleneck())
FUSE_RESIDUAL    = find_fuse_groups(AlexNetResidual())
FUSE_FIRE        = find_fuse_groups(AlexNetFire())
FUSE_FIRE_BYPASS = find_fuse_groups(AlexNetFireBypass())

# ── AlexNetFactorized: 1×3→3×1 Conv-BN-ReLU pairs (10 groups) ───────────────
# Stage 1 has two MaxPools (indices 6,7) to compensate for no strided conv
FUSE_FACTORIZED = [
    ["0","1","2"],["3","4","5"],      # Stage 1 (MaxPool×2 at 6,7)
    ["8","9","10"],["11","12","13"],  # Stage 2 (MaxPool at 14)
    ["15","16","17"],["18","19","20"],# Stage 3
    ["21","22","23"],["24","25","26"],# Stage 4
    ["27","28","29"],["30","31","32"],# Stage 5
]

# ── AlexNetGroupConv: one Conv-BN-ReLU per stage (5 groups) ──────────────────
FUSE_GROUPCONV = [
    ["0","1","2"],   # Stage 1 (MaxPool at 3)
    ["4","5","6"],   # Stage 2 (MaxPool at 7)
    ["8","9","10"],  # Stage 3
    ["11","12","13"],# Stage 4
    ["14","15","16"],# Stage 5
]

# ── AlexNetDepthwiseSep: DW+PW Conv-BN-ReLU pairs (10 groups) ────────────────
FUSE_DEPTHWISESEP = [
    ["0","1","2"],["3","4","5"],      # Stage 1 (MaxPool at 6)
    ["7","8","9"],["10","11","12"],   # Stage 2 (MaxPool at 13)
    ["14","15","16"],["17","18","19"],# Stage 3
    ["20","21","22"],["23","24","25"],# Stage 4
    ["26","27","28"],["29","30","31"],# Stage 5
]

MODEL_REGISTRY.clear()
register_model("alexnet_bottleneck",   AlexNetBottleneck,   fuse_map=FUSE_BOTTLENECK,   lr=1e-3)
register_model("alexnet_factorized",   AlexNetFactorized,   fuse_map=FUSE_FACTORIZED,   fuse_root_attr="features", lr=3e-4)
register_model("alexnet_groupconv",    AlexNetGroupConv,    fuse_map=FUSE_GROUPCONV,    fuse_root_attr="features", lr=1e-3)
register_model("alexnet_depthwisesep", AlexNetDepthwiseSep, fuse_map=FUSE_DEPTHWISESEP, fuse_root_attr="features", lr=1e-3)
register_model("alexnet_residual",     AlexNetResidual,     fuse_map=FUSE_RESIDUAL,     lr=3e-4)
register_model("alexnet_fire",         AlexNetFire,         fuse_map=FUSE_FIRE,         lr=1e-3)
register_model("alexnet_fire_bypass",  AlexNetFireBypass,   fuse_map=FUSE_FIRE_BYPASS,  lr=1e-3)
register_model("alexnet_se",           AlexNetSE,           fuse_map=[],                lr=3e-4)  # ponytail: QAT skipped — Sigmoid not fbgemm-compatible

print("Registered:", list(MODEL_REGISTRY))

## 5. FP32 training

In [6]:
fp32_training_results = {}

for name, spec in MODEL_REGISTRY.items():
    # Auto-detect resume checkpoint
    resume_from = auto_resume_path(SAVE_DIR, name)
    
    # Restore W&B run if resuming
    run_id = None
    if resume_from:
        meta_path = SAVE_DIR / f"{name}_meta.json"
        if meta_path.exists():
            run_id = json.loads(meta_path.read_text()).get("wandb_run_id")
    
    model_cfg = replace(fp32_cfg, lr=spec.get("lr", fp32_cfg.lr))
    print("=" * 72)
    print(f"Training: {name}  lr={model_cfg.lr}  epochs={model_cfg.epochs}")
    if resume_from:
        print(f"(Resuming from checkpoint)")
    print("=" * 72)

    model = spec["ctor"]().to(device)

    run = wandb.init(
        project="alexnet-phase3",
        group="fp32-phase3",
        name=f"{name}_fp32",
        id=run_id,
        resume="allow" if run_id else None,
        config={**asdict(model_cfg), "arch": name, "phase": "fp32",
                "num_classes": data_cfg.num_classes, "img_size": data_cfg.img_size,
                "dataset": "tiny-imagenet-200"},
        tags=["phase3", "compensation", "tiny-imagenet", "fp32"],
        mode="offline",
    )

    trainer = Trainer(
        model, train_loader, val_loader, model_cfg,
        device, SAVE_DIR, name, num_classes=data_cfg.num_classes,
        wandb_run=run,
        log_file=SAVE_DIR / f"{name}.log",
    )
    results = trainer.fit(resume_from=resume_from)
    wandb.finish()

    fp32_training_results[name] = results

    del model
    torch.cuda.empty_cache()

print("\nFP32 training complete.")

Training: alexnet_bottleneck  lr=0.001  epochs=100
(Resuming from checkpoint)


wandb: WARNING `resume` will be ignored since W&B syncing is set to `offline`. Starting a new run with run id ge83bymk.


Validation: 100%|██████████| 157/157 [00:01<00:00, 80.50it/s, loss=2.9743, top1=43.94%, top5=70.52%] 
Epoch  78/100 | train_loss=2.9119 train_acc=44.34% | val_loss=2.9743 val_acc=43.94% val_top5=70.52% | lr=1.15e-04 peak_mem=147MB time=27.8s
Early stopping at epoch 78

================= Run Summary =================
Model          : alexnet_bottleneck
Epochs         : 78
Best Val Top-1 : 44.68%
Best Val Top-5 : 70.95%
Final Val Top-1: 43.94%
Final Val Top-5: 70.52%
Best Val Loss  : inf
Total Time     : 00:00:27


epoch_time_s,▁
lr,▁
peak_gpu_mem_mb,▁
train_acc,▁
train_loss,▁
val_acc,▁
val_loss,▁
val_top5,▁
epoch_time_s,27.79146
lr,0.00011
peak_gpu_mem_mb,146.84814


Training: alexnet_factorized  lr=0.0003  epochs=100
(Resuming from checkpoint)


...

best_val_acc,▁▂▃▃▄▄▅▅▅▆▆▆▆▆▆▇▇▇▇█████
epoch_time_s,▇▇▆▆▆▅▆▅▅▆▅▅▄▄▅▄▄▄▄▅▅▇▇▇▇▇▇██▆▃▃▂▃▂▁▂▁▁
lr,█████████▇▇▇▇▇▇▇▇▆▆▆▆▆▅▅▅▅▄▄▄▄▃▃▃▃▂▂▂▁▁
peak_gpu_mem_mb,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
train_acc,▁▂▂▃▃▄▄▄▅▅▅▅▅▆▆▆▆▆▆▆▇▇▇▇▇▇▇▇▇▇▇████████
train_loss,█▇▆▅▅▅▄▄▄▄▃▃▃▃▃▃▃▂▂▂▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁
val_acc,▁▂▃▃▄▄▅▅▅▅▆▆▆▆▆▆▆▆▇▇▇▇▇█▇▇██▇██████████
val_loss,█▇▆▅▅▄▄▄▄▄▃▃▃▃▃▂▂▂▂▂▂▂▂▁▂▂▁▁▂▁▁▁▁▁▁▁▁▁▁
val_top5,▁▂▃▄▄▅▅▅▆▆▆▆▆▆▇▇▇▇▇▇▇▇▇██▇█████████████
best_val_acc,38.86
epoch_time_s,21.03819


Training: alexnet_se  lr=0.0003  epochs=100


Validation: 100%|██████████| 157/157 [00:01<00:00, 134.46it/s, loss=5.2987, top1=0.43%, top5=1.94%]
wandb: WARNING Symlinked 1 file into the W&B run directory; call wandb.save again to sync new files.
Epoch   1/100 | train_loss=5.2987 train_acc=0.46% | val_loss=5.2987 val_acc=0.43% val_top5=1.94% | lr=3.00e-04 peak_mem=1122MB time=49.1s
Validation: 100%|██████████| 157/157 [00:01<00:00, 139.01it/s, loss=5.2991, top1=0.31%, top5=1.93%]
Epoch   2/100 | train_loss=5.2985 train_acc=0.49% | val_loss=5.2991 val_acc=0.31% val_top5=1.93% | lr=3.00e-04 peak_mem=1122MB time=48.8s
Validation: 100%|██████████| 157/157 [00:01<00:00, 131.59it/s, loss=5.2993, top1=0.31%, top5=1.80%]
Epoch   3/100 | train_loss=5.2985 train_acc=0.51% | val_loss=5.2993 val_acc=0.31% val_top5=1.80% | lr=2.99e-04 peak_mem=1122MB time=48.9s
Validation: 100%|██████████| 157/157 [00:01<00:00, 143.99it/s, loss=5.2995, top1=0.31%, top5=1.80%]
Epoch   4/100 | train_loss=5.2985 train_acc=0.52% | val_loss=5.2995 val_acc=0.31% val

best_val_acc,▁
epoch_time_s,█▂▄▁▁▂
lr,█▇▆▅▃▁
peak_gpu_mem_mb,█▁▁▁▁▁
train_acc,▁▅██▅▅
train_loss,█▃▁▂▁▁
val_acc,█▁▁▁▁▁
val_loss,▁▃▅▆▇█
val_top5,██▃▃▃▁
best_val_acc,0.43
epoch_time_s,48.7678



FP32 training complete.


## 6. Quantization-Aware Training (QAT)

`AlexNetSE` is excluded: Sigmoid in SE blocks is not fbgemm-compatible.
All other models support QAT. QAT trains on GPU; `convert` and INT8 inference run on CPU (fbgemm).

In [7]:
QAT_SKIP = {"alexnet_se"}  # ponytail: Sigmoid in SE not fbgemm-compatible

qat_train_cfg = replace(
    fp32_cfg,
    epochs=qat_cfg.epochs,
    lr=qat_cfg.lr,
    weight_decay=qat_cfg.weight_decay,
    use_amp=False,  # AMP incompatible with fake-quant observers
)

qat_models = {}
qat_training_results = {}

for name in MODEL_REGISTRY:
    if name in QAT_SKIP:
        print(f"Skipping QAT for {name} (QAT_SKIP).")
        continue

    # Auto-detect resume checkpoint
    resume_from = auto_resume_path(SAVE_DIR, f"qat_{name}")
    
    # Restore W&B run if resuming
    run_id = None
    if resume_from:
        meta_path = SAVE_DIR / f"qat_{name}_meta.json"
        if meta_path.exists():
            run_id = json.loads(meta_path.read_text()).get("wandb_run_id")

    print("=" * 72)
    print(f"QAT fine-tuning: {name}")
    if resume_from:
        print(f"(Resuming from checkpoint)")
    print("=" * 72)

    qat_model = build_qat(name, save_dir=SAVE_DIR, device=device)
    cb = make_qat_callback(qat_cfg.freeze_bn_epoch, qat_cfg.disable_observer_epoch)

    run = wandb.init(
        project="alexnet-phase3",
        group="qat-phase3",
        name=f"{name}_qat",
        id=run_id,
        resume="allow" if run_id else None,
        config={
            **asdict(qat_train_cfg),
            "arch": name,
            "phase": "qat",
            "freeze_bn_epoch": qat_cfg.freeze_bn_epoch,
            "disable_observer_epoch": qat_cfg.disable_observer_epoch,
            "num_classes": data_cfg.num_classes, "img_size": data_cfg.img_size,
            "dataset": "tiny-imagenet-200",
        },
        tags=["phase3", "compensation", "tiny-imagenet", "qat"],
        mode="offline",
    )

    trainer = Trainer(
        qat_model, train_loader, val_loader, qat_train_cfg,
        device, SAVE_DIR, f"qat_{name}", num_classes=data_cfg.num_classes,
        wandb_run=run,
        epoch_callback=cb,
        log_file=SAVE_DIR / f"qat_{name}.log",
    )
    results = trainer.fit(resume_from=resume_from)
    wandb.finish()

    qat_training_results[name] = results
    qat_models[name] = qat_model.cpu()

    del qat_model
    torch.cuda.empty_cache()

print("\nQAT training complete.")

/home/rafael/Documents/alexnet_rafael/.venv/lib/python3.12/site-packages/torch/ao/quantization/observer.py:229: UserWarning: Please use quant_min and quant_max to specify the range for observers.                     reduce_range will be deprecated in a future release of PyTorch.
  warnings.warn(


QAT fine-tuning: alexnet_bottleneck


Validation: 100%|██████████| 157/157 [00:01<00:00, 112.73it/s, loss=2.9447, top1=44.68%, top5=70.75%]
wandb: WARNING Symlinked 1 file into the W&B run directory; call wandb.save again to sync new files.
Epoch   1/20 | train_loss=2.9918 train_acc=42.15% | val_loss=2.9447 val_acc=44.68% val_top5=70.75% | lr=9.94e-06 peak_mem=464MB time=30.3s
Validation: 100%|██████████| 157/157 [00:01<00:00, 110.21it/s, loss=2.9535, top1=44.37%, top5=70.76%]
Epoch   2/20 | train_loss=2.9858 train_acc=42.17% | val_loss=2.9535 val_acc=44.37% val_top5=70.76% | lr=9.76e-06 peak_mem=464MB time=29.4s
Validation: 100%|██████████| 157/157 [00:01<00:00, 111.73it/s, loss=2.9466, top1=44.67%, top5=70.84%]
Epoch   3/20 | train_loss=2.9796 train_acc=42.46% | val_loss=2.9466 val_acc=44.67% val_top5=70.84% | lr=9.46e-06 peak_mem=464MB time=29.7s
Validation: 100%|██████████| 157/157 [00:01<00:00, 111.84it/s, loss=2.9438, top1=44.42%, top5=70.84%]
Epoch   4/20 | train_loss=2.9238 train_acc=43.92% | val_loss=2.9438 val_ac

best_val_acc,▁█
epoch_time_s,▆▃▄▁▅▃▇▇▇█
lr,██▇▇▆▅▄▃▂▁
peak_gpu_mem_mb,███▁▁▁▁▁▁▁
train_acc,▁▁▂▇▇▇▇███
train_loss,█▇▇▂▂▂▁▁▁▁
val_acc,▇▁▆▂█▆▃▁▂▇
val_loss,▃█▄▃▁▂▁▃▇▁
val_top5,▁▁▂▂▂▃█▂▂▇
best_val_acc,44.76
epoch_time_s,30.63068


QAT fine-tuning: alexnet_factorized


...

/home/rafael/Documents/alexnet_rafael/.venv/lib/python3.12/site-packages/torch/ao/quantization/observer.py:229: UserWarning: Please use quant_min and quant_max to specify the range for observers.                     reduce_range will be deprecated in a future release of PyTorch.
  warnings.warn(


QAT fine-tuning: alexnet_gap


Validation: 100%|██████████| 157/157 [00:01<00:00, 114.38it/s, loss=3.1879, top1=39.54%, top5=65.42%]
wandb: WARNING Symlinked 1 file into the W&B run directory; call wandb.save again to sync new files.
Epoch   1/20 | train_loss=3.1647 train_acc=38.49% | val_loss=3.1879 val_acc=39.54% val_top5=65.42% | lr=9.94e-06 peak_mem=169MB time=24.7s
Validation: 100%|██████████| 157/157 [00:01<00:00, 119.26it/s, loss=3.1932, top1=39.43%, top5=65.21%]
Epoch   2/20 | train_loss=3.1425 train_acc=38.87% | val_loss=3.1932 val_acc=39.43% val_top5=65.21% | lr=9.76e-06 peak_mem=169MB time=24.6s
Validation: 100%|██████████| 157/157 [00:01<00:00, 116.01it/s, loss=3.1837, top1=39.69%, top5=65.25%]
Epoch   3/20 | train_loss=3.1293 train_acc=39.08% | val_loss=3.1837 val_acc=39.69% val_top5=65.25% | lr=9.46e-06 peak_mem=169MB time=24.7s
Validation: 100%|██████████| 157/157 [00:01<00:00, 116.50it/s, loss=3.1864, top1=39.59%, top5=65.40%]
Epoch   4/20 | train_loss=3.1273 train_acc=39.30% | val_loss=3.1864 val_ac

best_val_acc,▁▃██
epoch_time_s,▄▁▄▃▅▄█▅▂▇▁█
lr,██▇▇▆▆▅▄▄▃▂▁
peak_gpu_mem_mb,▁▁▁▁▁▁▁▁▁▁▁▁
train_acc,▁▃▄▅▆█▇▇▇▇██
train_loss,█▆▄▄▃▂▃▂▂▂▂▁
val_acc,▂▁▄▃█▆█▅▇▅▆▅
val_loss,▇█▅▆▂▆▁▃▃▄▂▄
val_top5,▄▁▁▃▆▂█▅█▆▆▇
best_val_acc,39.99
epoch_time_s,24.93332


Skipping QAT for alexnet_se (QAT_SKIP).

QAT training complete.


## 7. INT8 conversion & CPU evaluation

`torch.ao.quantization.convert` materialises true quantised ops and must run on CPU.

In [ ]:
val_loader_cpu = torch.utils.data.DataLoader(
    val_ds, batch_size=data_cfg.batch_size, shuffle=False, num_workers=0, pin_memory=False,
)
cpu_device = torch.device("cpu")

int8_models = {name: convert_to_int8(m.cpu().eval()) for name, m in qat_models.items()}

for name, m in int8_models.items():
    torch.save(m.state_dict(), SAVE_DIR / f"{name}.pth")
    compress_checkpoint(SAVE_DIR / f"{name}.pth")
print("INT8 conversion done.")

int8_metrics = {}
for name, m in int8_models.items():
    dummy_cfg = replace(fp32_cfg, use_amp=False)
    trainer = Trainer(
        m.cpu().eval(), val_loader_cpu, val_loader_cpu, dummy_cfg,
        cpu_device, SAVE_DIR, name, num_classes=data_cfg.num_classes,
    )
    metrics = trainer.evaluate(topk=(1, 5))
    int8_metrics[name] = metrics
    print(f"{name:22s} | loss={metrics['loss']:.4f} | top1={metrics['top1']:.2f}% | top5={metrics['top5']:.2f}%")

## 8. FP32 evaluation — reload best checkpoints

In [ ]:
CTORS = {
    "alexnet_bottleneck":   AlexNetBottleneck,
    "alexnet_factorized":   AlexNetFactorized,
    "alexnet_groupconv":    AlexNetGroupConv,
    "alexnet_depthwisesep": AlexNetDepthwiseSep,
    "alexnet_residual":     AlexNetResidual,
    "alexnet_fire":         AlexNetFire,
    "alexnet_fire_bypass":  AlexNetFireBypass,
    "alexnet_se":           AlexNetSE,
}

fp32_best = {name: load_best_model(name, ctor, SAVE_DIR, device) for name, ctor in CTORS.items()}

fp32_metrics = {}
for name, m in fp32_best.items():
    dummy_cfg = replace(fp32_cfg, use_amp=False)
    trainer = Trainer(
        m, train_loader, val_loader, dummy_cfg,
        device, SAVE_DIR, name, num_classes=data_cfg.num_classes,
    )
    metrics = trainer.evaluate(topk=(1, 5))
    fp32_metrics[name] = metrics
    print(f"{name:22s} | loss={metrics['loss']:.4f} | top1={metrics['top1']:.2f}% | top5={metrics['top5']:.2f}%")

## 9. Final comparison table

In [10]:
rows = []

for name, m in fp32_best.items():
    info = torchinfo.summary(m, input_size=(1, 3, data_cfg.img_size, data_cfg.img_size), verbose=0)
    rows.append({
        "model": name, "precision": "FP32",
        "top1_%": fp32_metrics[name]["top1"],
        "top5_%": fp32_metrics[name]["top5"],
        "loss":   fp32_metrics[name]["loss"],
        "params_M": info.total_params / 1e6,
        "size_MB":  disk_mb(SAVE_DIR / f"{name}_best.pth"),
    })

for name, m in int8_models.items():
    info = torchinfo.summary(fp32_best[name], input_size=(1, 3, data_cfg.img_size, data_cfg.img_size), verbose=0)
    rows.append({
        "model": f"{name}_INT8", "precision": "INT8",
        "top1_%": int8_metrics[name]["top1"],
        "top5_%": int8_metrics[name]["top5"],
        "loss":   int8_metrics[name]["loss"],
        "params_M": info.total_params / 1e6,
        "size_MB":  disk_mb(SAVE_DIR / f"{name}.pth"),
    })

df = build_comparison_table(rows)
df.to_csv(RESULTS_DIR / "final_comparison.csv", index=False)
df

,model,precision,top1_%,top5_%,loss,params_M,size_MB
0,alexnet_residual,FP32,48.012939,73.108339,2.247622,60.670280,694.414034
1,alexnet_bottleneck,FP32,44.622979,71.036339,2.362962,0.385000,4.491745
2,alexnet_depthwisesep,FP32,44.390011,69.445616,2.415972,0.313641,3.654016
3,alexnet_fire,FP32,43.976474,70.431584,2.409856,0.516152,5.989412
4,alexnet_factorized,FP32,42.890143,68.914163,2.497376,57.066760,653.148672
5,alexnet_gap,FP32,38.738528,64.831728,2.688579,2.302984,26.370630
6,alexnet_groupconv,FP32,29.181999,57.067955,3.078163,55.919752,639.990591
7,alexnet_se,FP32,0.500000,2.500000,5.298778,57.646288,659.752306
8,alexnet_residual_INT8,INT8,47.273704,72.499549,2.310971,60.670280,58.102711
9,alexnet_bottleneck_INT8,INT8,44.539082,71.014529,2.361130,0.385000,0.431780


In [ ]:
import json as _json

# --- Benchmark FP32 models (GPU) ---
fp32_benchmarks = {}
for name, m in fp32_best.items():
    dummy_cfg = replace(fp32_cfg, use_amp=False)
    t = Trainer(m, train_loader, val_loader, dummy_cfg, device, SAVE_DIR, name,
                num_classes=data_cfg.num_classes)
    fp32_benchmarks[name] = t.benchmark()
    print(f"{name:22s} FP32 | {fp32_benchmarks[name]['latency_ms_per_image']:.3f} ms/img")

# --- Benchmark INT8 models (CPU) ---
int8_benchmarks = {}
for name, m in int8_models.items():
    dummy_cfg = replace(fp32_cfg, use_amp=False)
    t = Trainer(m.cpu().eval(), val_loader_cpu, val_loader_cpu, dummy_cfg, cpu_device, SAVE_DIR, name,
                num_classes=data_cfg.num_classes)
    int8_benchmarks[name] = t.benchmark(warmup=50)
    print(f"{name:22s} INT8 | {int8_benchmarks[name]['latency_ms_per_image']:.3f} ms/img")

# --- FLOPs ---
fp32_flops = {}
for name, m in fp32_best.items():
    fp32_flops[name] = compute_flops(m.cpu().eval())
    print(f"{name:22s} | {fp32_flops[name]['macs']/1e9:.3f} GMACs")

# --- Per-model summary JSONs ---
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
for name in fp32_best:
    compress_checkpoint(SAVE_DIR / f"{name}_best.pth")
    info = torchinfo.summary(fp32_best[name], input_size=(1, 3, data_cfg.img_size, data_cfg.img_size), verbose=0)
    summary = make_run_summary(
        name=name,
        mode="fp32+qat+int8",
        fit_results=fp32_training_results.get(name, {}),
        fp32_eval=fp32_metrics[name],
        params_m=info.total_params / 1e6,
        fp32_size_mb=disk_mb(SAVE_DIR / f"{name}_best.pth"),
        int8_size_mb=disk_mb(SAVE_DIR / f"{name}.pth"),
        fp32_benchmark=fp32_benchmarks[name],
        flops_results=fp32_flops[name],
        int8_eval=int8_metrics.get(name),
        int8_benchmark=int8_benchmarks.get(name),
        fp32_gzip_mb=gzip_mb(SAVE_DIR / f"{name}_best.pth"),
        int8_gzip_mb=gzip_mb(SAVE_DIR / f"{name}.pth"),
    )
    out = RESULTS_DIR / f"{name}_summary.json"
    with open(out, "w") as f:
        _json.dump(summary, f, indent=2, default=str)
    print(f"Saved: {out.name}")

In [12]:
print("=" * 72)
print("RANKING BY TOP-1 ACCURACY (all precisions)")
print("=" * 72)
ranked = df.sort_values("top1_%", ascending=False).reset_index(drop=True)
for i, row in ranked.iterrows():
    print(f"{i+1:2d}. {row['model']:26s} [{row['precision']}] | "
          f"top1={row['top1_%']:6.2f}% | top5={row['top5_%']:6.2f}% | "
          f"params={row['params_M']:6.2f}M | size={row['size_MB']:6.2f}MB")

RANKING BY TOP-1 ACCURACY (all precisions)
 1. alexnet_residual           [FP32] | top1= 48.01% | top5= 73.11% | params= 60.67M | size=694.41MB
 2. alexnet_residual_INT8      [INT8] | top1= 47.27% | top5= 72.50% | params= 60.67M | size= 58.10MB
 3. alexnet_bottleneck         [FP32] | top1= 44.62% | top5= 71.04% | params=  0.39M | size=  4.49MB
 4. alexnet_bottleneck_INT8    [INT8] | top1= 44.54% | top5= 71.01% | params=  0.39M | size=  0.43MB
 5. alexnet_depthwisesep       [FP32] | top1= 44.39% | top5= 69.45% | params=  0.31M | size=  3.65MB
 6. alexnet_fire_INT8          [INT8] | top1= 44.30% | top5= 70.88% | params=  0.52M | size=  0.55MB
 7. alexnet_fire               [FP32] | top1= 43.98% | top5= 70.43% | params=  0.52M | size=  5.99MB
 8. alexnet_factorized         [FP32] | top1= 42.89% | top5= 68.91% | params= 57.07M | size=653.15MB
 9. alexnet_factorized_INT8    [INT8] | top1= 42.60% | top5= 67.80% | params= 57.07M | size= 54.68MB
10. alexnet_depthwisesep_INT8  [INT8] | top1= 41

## 10. Persist experiment summary

In [13]:
create_results_summary(
    results={
        "fp32_metrics": fp32_metrics,
        "int8_metrics": int8_metrics,
        "fp32_training_results": fp32_training_results,
        "qat_training_results": qat_training_results,
    },
    config=asdict(fp32_cfg),
    output_path=RESULTS_DIR / "experiment_summary.json",
)

## W&B — syncing offline runs

All runs were saved locally with `mode="offline"`. Two run groups are tracked:
- **`fp32-phase3`** — one run per architecture, FP32 training
- **`qat-phase3`** — one run per architecture (excl. `alexnet_se`), QAT fine-tuning

When ready, sync to the W&B dashboard from a terminal:

```bash
wandb sync --sync-all
```